In [2]:

import pandas as pd
from pathlib import Path

# Find the dataset in the current folder
dataset_path = Path.cwd() / "breast_cancer.csv"

df = pd.read_csv(dataset_path)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

display(df.head())


Dataset loaded successfully!
Dataset shape: (569, 31)


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


In [3]:

import pandas as pd
import json
import joblib

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Project paths
PROJECT_ROOT = Path.cwd()
MODEL_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"

MODEL_DIR.mkdir(exist_ok=True)
REPORTS_DIR.mkdir(exist_ok=True)

# Load dataset
df = pd.read_csv(PROJECT_ROOT / "breast_cancer.csv")

# Separate features and target
X = df.drop(columns=["target"])
y = df["target"]

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Build preprocessing + prediction pipeline
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

# Train the model
model.fit(X_train, y_train)

# Evaluate on unseen test data
y_pred = model.predict(X_test)

metrics = {
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "precision": float(precision_score(y_test, y_pred, zero_division=0)),
    "recall": float(recall_score(y_test, y_pred, zero_division=0)),
    "f1_score": float(f1_score(y_test, y_pred, zero_division=0))
}

# Save trained pipeline
joblib.dump(model, MODEL_DIR / "breast_cancer_pipeline.joblib")

# Save metrics
with open(REPORTS_DIR / "day5_metrics.json", "w") as file:
    json.dump(metrics, file, indent=4)

print("Pipeline trained and saved successfully!")
print("Metrics:", metrics)
print("Model saved to:", MODEL_DIR / "breast_cancer_pipeline.joblib")
print("Metrics saved to:", REPORTS_DIR / "day5_metrics.json")


Pipeline trained and saved successfully!
Metrics: {'accuracy': 0.9824561403508771, 'precision': 0.9861111111111112, 'recall': 0.9861111111111112, 'f1_score': 0.9861111111111112}
Model saved to: c:\Users\Work\Desktop\Reproducible tabular prediction workflow\Day5-End to end ML capstone\models\breast_cancer_pipeline.joblib
Metrics saved to: c:\Users\Work\Desktop\Reproducible tabular prediction workflow\Day5-End to end ML capstone\reports\day5_metrics.json


In [4]:

import joblib
import pandas as pd
from pathlib import Path

# Load the saved model and dataset
PROJECT_ROOT = Path.cwd()
model = joblib.load(
    PROJECT_ROOT / "models" / "breast_cancer_pipeline.joblib"
)
df = pd.read_csv(PROJECT_ROOT / "breast_cancer.csv")

X = df.drop(columns=["target"])

# Test 1: Valid input
valid_input = X.iloc[[0]]
prediction = model.predict(valid_input)

print("Test 1 - Valid input: PASSED")
print("Predicted class:", prediction[0])

# Test 2: Single-row input
assert len(model.predict(X.iloc[[0]])) == 1
print("Test 2 - Single-row input: PASSED")

# Test 3: Missing required feature
try:
    invalid_input = valid_input.drop(columns=[X.columns[0]])
    model.predict(invalid_input)
    print("Test 3 - FAILED: Missing feature was accepted")
except ValueError:
    print("Test 3 - Missing feature: PASSED (rejected)")

# Test 4: Extra unexpected feature
try:
    invalid_input = valid_input.copy()
    invalid_input["unexpected_feature"] = 123
    model.predict(invalid_input)
    print("Test 4 - FAILED: Extra feature was accepted")
except ValueError:
    print("Test 4 - Extra feature: PASSED (rejected)")


Test 1 - Valid input: PASSED
Predicted class: 0
Test 2 - Single-row input: PASSED
Test 3 - Missing feature: PASSED (rejected)
Test 4 - Extra feature: PASSED (rejected)


In [5]:

import json
import sklearn
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd()
MODEL_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"

# Model metadata
metadata = {
    "project": "Breast Cancer Prediction",
    "model_type": "Logistic Regression",
    "preprocessing": [
        "Median imputation",
        "StandardScaler"
    ],
    "target_column": "target",
    "class_mapping": {
        "0": "malignant",
        "1": "benign"
    },
    "test_size": 0.20,
    "random_state": 42,
    "model_parameters": {
        "max_iter": 1000,
        "random_state": 42
    },
    "metrics": {
        "accuracy": 0.9824561403508771,
        "precision": 0.9861111111111112,
        "recall": 0.9861111111111112,
        "f1_score": 0.9861111111111112
    },
    "python_library_versions": {
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__
    }
}

metadata_path = REPORTS_DIR / "day5_model_metadata.json"

with open(metadata_path, "w") as file:
    json.dump(metadata, file, indent=4)

print("Model metadata saved successfully!")
print("Location:", metadata_path)


Model metadata saved successfully!
Location: c:\Users\Work\Desktop\Reproducible tabular prediction workflow\Day5-End to end ML capstone\reports\day5_model_metadata.json
